In [1]:
import os
import pathlib
import sys

here = pathlib.Path.cwd()
ROOT = here.parents[2] if here.name == "day02" else here
os.chdir(ROOT)
SANDBOX = ROOT / "sandbox" / "w8" / "day02"

print("프로젝트 루트  :", ROOT)

if str(ROOT / 'backend') not in sys.path:
    sys.path.insert(0, str(ROOT / 'backend'))

SAMPLES = ROOT / "sandbox" / "w5" / "day01" / "samples"
FORMATS = SAMPLES / "_formats"

프로젝트 루트  : /Users/pyoyoung-gyu/Desktop/Personal Project/한화아카데미/AI 서비스 백엔드 프로그래밍 실무/hanwha-agent/agent_practice


In [2]:
from anthropic.types import Message
BASE = {'id': 'msg_01FAKE', 'type': 'message', 'role': 'assistant',
        'model': 'claude-haiku-4-5', 'stop_sequence': None, 'usage': {'input_tokens': 0, 'output_tokens': 0}}

ROUNDS = [{**BASE, 'stop_reason': 'tool_use', 'content': [
            {'type': 'text', 'text': '규정을 먼저 찾아보겠습니다.'},
            {'type': 'tool_use', 'id': 'toolu_01ABC', 'name': 'search_documents', 'input': {
                'query': '부산 출장 숙박비 한도'}}]},
        {**BASE, 'stop_reason': 'tool_use', 'content': [
            {'type': 'tool_use', 'id': 'toolu_02DEF', 'name': 'create_draft', 'input': {
                'doc_type': '국내출장 신청서',
                'fields': {'출장지': '부산', '기간': '10/13~15'}}}]},
        {**BASE, 'stop_reason': 'end_turn', 'content': [{'type': 'text', 'text': '신청서 초안을 만들었습니다.'}]}]

CANNED = [Message.model_validate(r) for r in ROUNDS]

for i, m in enumerate(CANNED, 1):
    kinds = [b.type for b in m.content]
    used = [b.name for b in m.content if b.type == 'tool_use']
    print(f"{i}바퀴  {m.stop_reason:<9}  {str(kinds):<22}→ {(used[0] if used else '(도구 없음)')}")


1바퀴  tool_use   ['text', 'tool_use']  → search_documents
2바퀴  tool_use   ['tool_use']          → create_draft
3바퀴  end_turn   ['text']              → (도구 없음)


In [3]:
messages = [{'role': 'user', 'content': '질문 한 줄'}]
for m in CANNED:
    calls = [b for b in m.content if b.type == 'tool_use']
    if not calls:
        break
    messages.append({'role': 'assistant', 'content': [b.type for b in m.content]})
    messages.append({'role': 'user', 'content': ['tool_result'] * len(calls)})
for i, m in enumerate(messages, 1):
    print(i, f"{m['role']:<10}", m['content'])


1 user       질문 한 줄
2 assistant  ['text', 'tool_use']
3 user       ['tool_result']
4 assistant  ['tool_use']
5 user       ['tool_result']


In [4]:
import importlib
from app.services import chat_service as cs

from app.db.session import session_scope
from app.models import Run
from app.services.ids import next_run_id

user = {'name': '김민준', 'dept': '인프라사업부 2팀', 'clearance': '일반'}

with session_scope() as session:
    rid = next_run_id(session)
    session.add(Run(id=rid, user_id=1, question='부산 출장 신청서', mode='mock'))
ours = {'search_documents': {'user': user}, 'create_draft': {'user': user, 'run_id': rid}}
ARGS = {'search_documents': {'query': '부산 출장 숙박비 한도'}, 'create_draft': {'doc_type': '국내출장 신청서', 'fields': {'출장지': '부산', '기간': '10/13~15'}}}
RETURNED = {}
for name, tool_run in cs.TOOL_RUNNERS.items():
    RETURNED[name] = tool_run(**ARGS[name], **ours[name])
    print(f'{name:<17}: {sorted(RETURNED[name])}')

last_out: dict = {}
NAMES = {'search_documents': '검색', 'create_draft': '초안'}
for name in ('search_documents', 'create_draft'):
    last_out = RETURNED[name]
    kept = len(last_out.get('sources', []))
    mark = '   ← ★★★ 사라졌다' if kept == 0 else ''
    print(f'{NAMES[name]} 도구를 돈 뒤 : sources {kept}건{mark}')


12:30:53 INFO     sentence_transformers.base.model : No device provided, using mps
12:30:53 INFO     sentence_transformers.base.model : Loading SentenceTransformer model from ../models/bge-m3.


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

12:30:55 INFO     app.integrations.local_embed : 로컬 임베딩 모델 준비: ../models/bge-m3 / 1024차원


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

12:30:55 INFO     app.rag.retriever : 정책으로 제외: 국내출장 여비 규정 v1.1 - 만료 버전/국내출장 여비 규정 v1.1 - 만료 버전/국내출장 여비 규정 v1.1 - 만료 버전/국내출장 여비 규정 v1.1 - 만료 버전/법인카드 사용 지침 v2.0 - 만료 버전
12:30:55 INFO     sentence_transformers.base.model : No device provided, using mps
12:30:55 INFO     sentence_transformers.base.model : No modules.json found for ../models/bge-reranker-v2-m3, initializing a new CrossEncoder model.


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

12:30:56 INFO     app.integrations.local_rerank : 로컬 리랭커 준비 : ../models/bge-reranker-v2-m3, 최대 512토큰
search_documents : ['best_score', 'passed', 'sources', 'steps', 'threshold', 'version_note']
create_draft     : ['body', 'doc_type', 'fields']
검색 도구를 돈 뒤 : sources 3건
초안 도구를 돈 뒤 : sources 0건   ← ★★★ 사라졌다


In [5]:
importlib.reload(cs)
last_out, made = ({}, None)
for name in ('search_documents', 'create_draft'):
    out = RETURNED[name]
    if name == cs.create_draft.NAME:
        made = out
    else:
        last_out = out
print(f"sources   : {len(last_out.get('sources', []))}건   : 살아남았다")
print(f'초안 있음 : {made is not None}')
print(f'초안 칸   : {sorted(made)}')


sources   : 3건   : 살아남았다
초안 있음 : True
초안 칸   : ['body', 'doc_type', 'fields']
